In [113]:
from utils import *

import torch as th
import torch.nn as nn
import torch.nn.functional as F

import importlib
import data_handler

importlib.reload(data_handler)

import tqdm


<class 'torch.Tensor'>


In [145]:
class FCN(nn.Module):
    def __init__(self):
        super(FCN, self).__init__()
        
        """
        VGG-16 
        """
        
        # Regular convolutional layers, same as VGG-16 
        self.layers = nn.Sequential(
            # 1 section (64)
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=100), nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            
            # 2 section (128)
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, ceil_mode=True),
            
            # 3 section (256)
            nn.Conv2d(128, 256, 3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, ceil_mode=True),
            
            # 4 section (512)
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, ceil_mode=True),
            
            # 5 section (512)
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2, ceil_mode=True),
        )
        
        # Classification layers
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        x = self.layers(x)
        x = self.classifier(x)
        return x
    
if __name__ == "__main__":
    model = FCN()
    print(model)

FCN(
  (layers): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(100, 100))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [148]:
device = th.device("cuda" if th.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = FCN().to(device)


def train():
    """
    Function to train the model. 
    All the hyperparameters are defined in this function internally.
    Loss function: BCEWithLogitsLoss
    Optimizer: Adam
    Epochs: 40
    Learning Rate: 0.0001
    """
    # params
    epochs = 10
    lr = 0.0001
    # Loss function
    #loss_func = nn.BCEWithLogitsLoss()
    loss_func = nn.CrossEntropyLoss()

    # Optimizer and model inits
    optimizer = th.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in tqdm.tqdm(range(epochs)):
        for images, labels in data_handler.tr_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)

            # Calculate the loss
            loss = loss_func(outputs, labels)

            # Backward pass and gradient update
            loss.backward()
            optimizer.step()
            
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

Device: cpu


In [150]:
train()

  0%|          | 0/10 [00:07<?, ?it/s]


KeyboardInterrupt: 